In [14]:
import numpy as np
import os
import pandas as pd
import pickle
project_dir = os.path.expanduser('../')
load_dir_vf = project_dir + "/data/value_functions"
load_dir_π = project_dir + "/data/policies"


In [15]:
SA_mask = pd.read_pickle(project_dir + "data/behavior_policy/SA_mask_shifted.pkl")
if isinstance(SA_mask, pd.DataFrame):
    SA_mask = SA_mask.values
    
for fname in sorted(os.listdir(load_dir_vf)):
    if fname.endswith(".npz"):
        var_name = fname.replace(".npz", "")
        globals()[var_name] = np.load(os.path.join(load_dir_vf, fname))["data"]

for fname in sorted(os.listdir(load_dir_π)):
    if fname.endswith(".pkl"):
        var_name = fname.replace(".pkl", "")
        with open(os.path.join(load_dir_π, fname), "rb") as f:
            globals()[var_name] = pickle.load(f)

γ = 1

nS = 750
nA = 25

nS_term = 2
S_survival = 750
S_death = 751

nS_total = nS + nS_term

### Checking Propositions
#### Checking Proposition 2
Proposition 2 holds, which is expected (proposition 2 doesn't require guaranteed absorption)

V_plus + V_minus = V_mixed


In [16]:
np.allclose(V_plus + V_minus, V_mixed, atol=1e-6, rtol=0)

True

Q_plus + Q_minus = Q_mixed

In [17]:
valid_mask = ~np.isnan(Q_plus) & ~np.isnan(Q_minus) & ~np.isnan(Q_mixed)
diff = np.abs((Q_plus + Q_minus) - Q_mixed)
print(np.all(diff[valid_mask] <= 1e-6))

True


#### Checking Proposition 4

V_plus - V_minus = 1

In [18]:
np.allclose(V_plus - V_minus, 1.0, atol=1e-7, rtol=0)
np.where(~np.isclose(V_plus - V_minus, 1.0, atol=1e-7, rtol=0))

(array([271, 330, 482]),)

In [19]:
for target_states in [271, 330, 482]:
    print(V_plus[target_states] - V_minus[target_states])

0.9999998590580437
0.0
0.0


#### Checking Proposition 5

Q_plus - Q_minus = 1

In [20]:
tol = 1e-6
valid_mask = ~np.isnan(Q_plus) & ~np.isnan(Q_minus)
diff = (Q_plus - Q_minus) - 1.0
fail_mask = valid_mask & (np.abs(diff) > tol)
bad_indices = np.argwhere(fail_mask)
n_valid = np.count_nonzero(valid_mask)
n_fail  = len(bad_indices)
print(f"Checked {n_valid} valid (s,a) pairs → {n_fail} violated Proposition 4 "
      f"(Q⁺−Q⁻ ≈ 1).")

for s, a in bad_indices:
    print(f"State {s:3d}, Action {a:2d} | "
          f"Q⁺={Q_plus[s,a]:.6f}, Q⁻={Q_minus[s,a]:.6f}, "
          f"Δ={Q_plus[s,a]-Q_minus[s,a]:.6f}")


Checked 3834 valid (s,a) pairs → 5 violated Proposition 4 (Q⁺−Q⁻ ≈ 1).
State  82, Action  5 | Q⁺=0.934426, Q⁻=-0.049180, Δ=0.983607
State 116, Action  5 | Q⁺=0.962963, Q⁻=-0.000000, Δ=0.962963
State 330, Action  7 | Q⁺=0.000000, Q⁻=-0.250000, Δ=0.250000
State 330, Action 18 | Q⁺=0.000000, Q⁻=0.000000, Δ=0.000000
State 482, Action  9 | Q⁺=0.000000, Q⁻=0.000000, Δ=0.000000


#### Comparing policies
In proposition 6, we expected when all three assumptions hold, all policies should be equivalent.

**gamma = 1**

π_plus vs. π_minus

In [21]:
def compare_policy(π1, π2, name_π1, name_π2):
    count = 0
    for s in range(nS):
        if not np.array_equal(π1[s], π2[s]):
            count += 1
            print(f"Different policies between {name_π1} and {name_π2} at state {s}")
            print(f"{name_π1}: ", π1[s])
            print(f"{name_π2}: ", π2[s])
    print("Number of states with different policy: ", count)

In [22]:
# Comparing π_plus and π_minus, print different states;
compare_policy(π_plus, π_minus, "π_plus", "π_minus")

Different policies between π_plus and π_minus at state 116
π_plus:  [0]
π_minus:  [0, 5]
Different policies between π_plus and π_minus at state 330
π_plus:  [7, 18]
π_minus:  [18]
Number of states with different policy:  2


π_plus vs. π_mixed

In [23]:
# Comparing π_plus and π_minus, print different states;
compare_policy(π_plus, π_mixed, "π_plus", "π_mixed")

Different policies between π_plus and π_mixed at state 51
π_plus:  [5, 8, 10]
π_mixed:  [5, 8]
Different policies between π_plus and π_mixed at state 294
π_plus:  [10, 15]
π_mixed:  [15]
Different policies between π_plus and π_mixed at state 330
π_plus:  [7, 18]
π_mixed:  [18]
Different policies between π_plus and π_mixed at state 478
π_plus:  [5, 10, 14, 15]
π_mixed:  [10, 14, 15]
Different policies between π_plus and π_mixed at state 569
π_plus:  [0, 1]
π_mixed:  [1]
Number of states with different policy:  5


π_minus vs. π_mixed

In [24]:
# Comparing π_minus and π_mixed, print different states;
compare_policy(π_minus, π_mixed, "π_minus", "π_mixed")

Different policies between π_minus and π_mixed at state 51
π_minus:  [5, 8, 10]
π_mixed:  [5, 8]
Different policies between π_minus and π_mixed at state 116
π_minus:  [0, 5]
π_mixed:  [0]
Different policies between π_minus and π_mixed at state 294
π_minus:  [10, 15]
π_mixed:  [15]
Different policies between π_minus and π_mixed at state 478
π_minus:  [5, 10, 14, 15]
π_mixed:  [10, 14, 15]
Different policies between π_minus and π_mixed at state 569
π_minus:  [0, 1]
π_mixed:  [1]
Number of states with different policy:  5


**gamma = 0.99**

In [25]:
# Comparing π_plus and π_minus, print different states;
compare_policy(π_plus_gamma_99, π_minus_gamma_99, "π_plus_gamma_99", "π_minus_gamma_99")

Different policies between π_plus_gamma_99 and π_minus_gamma_99 at state 0
π_plus_gamma_99:  [0]
π_minus_gamma_99:  [0, 6, 7, 8, 10, 15]
Different policies between π_plus_gamma_99 and π_minus_gamma_99 at state 1
π_plus_gamma_99:  [0]
π_minus_gamma_99:  [0, 5, 10, 15]
Different policies between π_plus_gamma_99 and π_minus_gamma_99 at state 2
π_plus_gamma_99:  [0]
π_minus_gamma_99:  [0, 6, 7, 8, 10]
Different policies between π_plus_gamma_99 and π_minus_gamma_99 at state 3
π_plus_gamma_99:  [8]
π_minus_gamma_99:  [6, 8, 9, 14, 15]
Different policies between π_plus_gamma_99 and π_minus_gamma_99 at state 4
π_plus_gamma_99:  [8]
π_minus_gamma_99:  [0, 7, 8, 9, 10, 15]
Different policies between π_plus_gamma_99 and π_minus_gamma_99 at state 5
π_plus_gamma_99:  [0]
π_minus_gamma_99:  [6, 8, 9, 15]
Different policies between π_plus_gamma_99 and π_minus_gamma_99 at state 6
π_plus_gamma_99:  [0]
π_minus_gamma_99:  [0, 5, 7, 10, 15]
Different policies between π_plus_gamma_99 and π_minus_gamma_99 

In [26]:
# Comparing π_plus and π_minus, print different states;
compare_policy(π_plus_gamma_99, π_mixed_gamma_99, "π_plus_gamma_99", "π_mixed__gamma_99")

Different policies between π_plus_gamma_99 and π_mixed__gamma_99 at state 41
π_plus_gamma_99:  [5]
π_mixed__gamma_99:  [0]
Different policies between π_plus_gamma_99 and π_mixed__gamma_99 at state 65
π_plus_gamma_99:  [0]
π_mixed__gamma_99:  [15]
Different policies between π_plus_gamma_99 and π_mixed__gamma_99 at state 106
π_plus_gamma_99:  [0]
π_mixed__gamma_99:  [6]
Different policies between π_plus_gamma_99 and π_mixed__gamma_99 at state 167
π_plus_gamma_99:  [10]
π_mixed__gamma_99:  [5]
Different policies between π_plus_gamma_99 and π_mixed__gamma_99 at state 179
π_plus_gamma_99:  [0]
π_mixed__gamma_99:  [6]
Different policies between π_plus_gamma_99 and π_mixed__gamma_99 at state 246
π_plus_gamma_99:  [5]
π_mixed__gamma_99:  [15]
Different policies between π_plus_gamma_99 and π_mixed__gamma_99 at state 308
π_plus_gamma_99:  [0]
π_mixed__gamma_99:  [7]
Different policies between π_plus_gamma_99 and π_mixed__gamma_99 at state 320
π_plus_gamma_99:  [0]
π_mixed__gamma_99:  [9]
Differe

In [27]:
# Comparing π_minus and π_mixed, print different states;
compare_policy(π_minus_gamma_99, π_mixed_gamma_99, "π_minus_gamma_99", "π_mixed__gamma_99")

Different policies between π_minus_gamma_99 and π_mixed__gamma_99 at state 0
π_minus_gamma_99:  [0, 6, 7, 8, 10, 15]
π_mixed__gamma_99:  [0]
Different policies between π_minus_gamma_99 and π_mixed__gamma_99 at state 1
π_minus_gamma_99:  [0, 5, 10, 15]
π_mixed__gamma_99:  [0]
Different policies between π_minus_gamma_99 and π_mixed__gamma_99 at state 2
π_minus_gamma_99:  [0, 6, 7, 8, 10]
π_mixed__gamma_99:  [0]
Different policies between π_minus_gamma_99 and π_mixed__gamma_99 at state 3
π_minus_gamma_99:  [6, 8, 9, 14, 15]
π_mixed__gamma_99:  [8]
Different policies between π_minus_gamma_99 and π_mixed__gamma_99 at state 4
π_minus_gamma_99:  [0, 7, 8, 9, 10, 15]
π_mixed__gamma_99:  [8]
Different policies between π_minus_gamma_99 and π_mixed__gamma_99 at state 5
π_minus_gamma_99:  [6, 8, 9, 15]
π_mixed__gamma_99:  [0]
Different policies between π_minus_gamma_99 and π_mixed__gamma_99 at state 6
π_minus_gamma_99:  [0, 5, 7, 10, 15]
π_mixed__gamma_99:  [0]
Different policies between π_minus_g

#### Comparing policies between gamma = 1 and gamma = 0.99

In [28]:
compare_policy(π_plus, π_plus_gamma_99, "π_plus", "π_plus_gamma_99")

Different policies between π_plus and π_plus_gamma_99 at state 0
π_plus:  [0, 6, 7, 8, 10, 15]
π_plus_gamma_99:  [0]
Different policies between π_plus and π_plus_gamma_99 at state 1
π_plus:  [0, 5, 10, 15]
π_plus_gamma_99:  [0]
Different policies between π_plus and π_plus_gamma_99 at state 2
π_plus:  [0, 6, 7, 8, 10]
π_plus_gamma_99:  [0]
Different policies between π_plus and π_plus_gamma_99 at state 3
π_plus:  [6, 8, 9, 14, 15]
π_plus_gamma_99:  [8]
Different policies between π_plus and π_plus_gamma_99 at state 4
π_plus:  [0, 7, 8, 9, 10, 15]
π_plus_gamma_99:  [8]
Different policies between π_plus and π_plus_gamma_99 at state 5
π_plus:  [6, 8, 9, 15]
π_plus_gamma_99:  [0]
Different policies between π_plus and π_plus_gamma_99 at state 6
π_plus:  [0, 5, 7, 10, 15]
π_plus_gamma_99:  [0]
Different policies between π_plus and π_plus_gamma_99 at state 7
π_plus:  [0, 6, 7, 9, 10, 15]
π_plus_gamma_99:  [0]
Different policies between π_plus and π_plus_gamma_99 at state 8
π_plus:  [0, 5, 8, 10,

In [29]:
compare_policy(π_minus, π_minus_gamma_99, "π_minus", "π_minus_gamma_99")

Different policies between π_minus and π_minus_gamma_99 at state 635
π_minus:  [0, 10, 15]
π_minus_gamma_99:  [0, 7, 10, 15]
Number of states with different policy:  1


In [30]:
compare_policy(π_mixed, π_mixed_gamma_99, "π_mixed", "π_mixed_gamma_99")

Different policies between π_mixed and π_mixed_gamma_99 at state 0
π_mixed:  [0, 6, 7, 8, 10, 15]
π_mixed_gamma_99:  [0]
Different policies between π_mixed and π_mixed_gamma_99 at state 1
π_mixed:  [0, 5, 10, 15]
π_mixed_gamma_99:  [0]
Different policies between π_mixed and π_mixed_gamma_99 at state 2
π_mixed:  [0, 6, 7, 8, 10]
π_mixed_gamma_99:  [0]
Different policies between π_mixed and π_mixed_gamma_99 at state 3
π_mixed:  [6, 8, 9, 14, 15]
π_mixed_gamma_99:  [8]
Different policies between π_mixed and π_mixed_gamma_99 at state 4
π_mixed:  [0, 7, 8, 9, 10, 15]
π_mixed_gamma_99:  [8]
Different policies between π_mixed and π_mixed_gamma_99 at state 5
π_mixed:  [6, 8, 9, 15]
π_mixed_gamma_99:  [0]
Different policies between π_mixed and π_mixed_gamma_99 at state 6
π_mixed:  [0, 5, 7, 10, 15]
π_mixed_gamma_99:  [0]
Different policies between π_mixed and π_mixed_gamma_99 at state 7
π_mixed:  [0, 6, 7, 9, 10, 15]
π_mixed_gamma_99:  [0]
Different policies between π_mixed and π_mixed_gamma_99

#### Quick Summaries on Differences

**γ = 1**

| Comparison | # of Different States | Interpretation |
|-------------|----------------------|----------------|
| π⁺ vs. π⁻ | **2** |  |
| π⁺ vs. π+- | **5** |  |
| π⁻ vs. π+- | **5** |  |

**γ = 0.99**

| Comparison | # of Different States | Interpretation |
|-------------|----------------------|----------------|
| π⁺ vs. π⁻ | **700** |  |
| π⁺ vs. π+- | **20** |  |
| π⁻ vs. π+- | **695** |  |

**γ = 1 → γ = 0.99**

| Policy | # of Changed States | Interpretation |
|---------|--------------------|----------------|
| π⁺ | **698** |  |
| π⁻ | **1** |  |
| π+- | **694** |  |



3 interesting findings:
- for gamma = 1, intra policy difference is small. for gamma = 0.99, intra policy difference is big (π+ vs. π+- is relatively small). Expected; temporality.
- for gamma = 0.99, intra policy difference is relatively big but not for π+ vs. π+-. Expected; π- adds only death avoidance which is already encoded in π+.
- when gamma changes from 1 -> 0.99, π+ and π+- changed dramatically (nS almost 700), but π- almost did not change. Expected; gamma = 0.99 introduces temporality, which break former ties by prioritizing faster recovery.

To conclude, we can't confidently conclude that π+ is the "best". But it does optimizes discounted time to survival, which is survival oriented + temporality discounted, which might be useful in some cases. Same thing applies to gamma, use 0.99 if we want to include faster recovery, gamma = 1 if we only care about eventual outcome, regardless of when it happens.<br>
e.g. In chronic disease treatment, the trajectory matter as much as the endpoint, and faster recovery is not "always preferred".